# 🧠 Full Dataset Modeling - Real Estate Price Prediction

"""
This script trains various regression models on the full real estate dataset (28 million rows):

✅ Linear Regression (Baseline)
✅ Random Forest Regressor
✅ XGBoost Regressor
✅ Gradient Boosting Regressor (GBR)
✅ LightGBM Regressor
✅ CatBoost Regressor

🔧 Includes:
- Data loading from GCS
- Train/test split
- Model training and evaluation (MAE, RMSE, R²)
- Cross-Validation
- Model comparison and saving results

Author: Nguyễn Minh Trí

Date: April 2025
"""


In [ ]:
from google.colab import files
uploaded = files.upload()

KeyboardInterrupt: 

In [ ]:
!pip install --upgrade --force-reinstall catboost

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 7.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 4.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.5/102.5 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.7/98.7 MB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 92.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 99.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.1/47.1 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 102.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.8/14.8 MB 93.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.6/37.6 MB 52.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 326.2/326.2 kB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from google.cloud import storage
import os
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "/content/gcp-key.json"
print("Environment variable set!")
# Utility function for evaluation
def evaluate_model(name, model, X_test, y_test):
    y_pred = model.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    print(f"{name} - MAE: {mae:.4f}, RMSE: {rmse:.4f}, R²: {r2:.4f}")
    return name, mae, rmse, r2

# Prepare to download data from GCS
bucket_name = "boothill2001-dataset"
gcs_path = "uk_property_data/processed/cleaned_real_estate_full.csv"
local_path = "data/processed/cleaned_real_estate_full.csv"

# Download data function
def download_from_gcs(gcs_path, local_path):
    client = storage.Client()
    bucket = client.bucket(bucket_name)
    if not os.path.exists(local_path):
        blob = bucket.blob(gcs_path)
        os.makedirs(os.path.dirname(local_path), exist_ok=True)
        blob.download_to_filename(local_path)
        print("Data downloaded successfully!")
    else:
        print("Data already exists locally.")

# Download the full dataset
download_from_gcs(gcs_path, local_path)

# Load the data
df = pd.read_csv(local_path)
print("Data loaded! Shape:", df.shape)

# Cleaning categorical data
print("Cleaning categorical variables...")
for col in ['Property_Type', 'Urban_Rural', 'Season', 'Old/New', 'Duration', 'PPDCategory_Type', 'Record_Status']:
    if df[col].dtype == 'object':
        df[col] = df[col].replace(['Y', 'N', 'Unknown', 'BA'], 'Other')
        df[col] = df[col].astype(str)
print("Categorical variables cleaned.")

# One-Hot Encoding for categorical variables
df = pd.get_dummies(df, columns=['Property_Type', 'Urban_Rural', 'Season', 'Old/New', 'Duration', 'PPDCategory_Type', 'Record_Status'], drop_first=True)

# Feature and target selection
all_columns = list(df.columns)
exclude_columns = ['Transaction_ID', 'Date_of_Transfer', 'Postcode', 'PAON', 'Street', 'Locality', 'Town/City', 'District', 'County', 'Price', 'Region_Code']
features = [col for col in all_columns if col not in exclude_columns and col != 'Log_Price']
target = 'Log_Price'

# Train-test split
X = df[features]
y = df[target]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Data split into train and test sets.")

# 💾 Save feature names for Streamlit inference
import json
with open("feature_names.json", "w") as f:
    json.dump(list(X_train.columns), f)
print("✅ feature_names.json saved!")

# Train models and evaluate
models = {
    'Linear Regression': LinearRegression(),
    'XGBoost': XGBRegressor(random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(random_state=42),
    'LightGBM': LGBMRegressor(random_state=42),
    'CatBoost': CatBoostRegressor(random_state=42, verbose=0),
}

results = []
for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train, y_train)
    results.append(evaluate_model(name, model, X_test, y_test))

# Display results
results_df = pd.DataFrame(results, columns=['Model', 'MAE', 'RMSE', 'R²'])
print("\nModel Performance Comparison:")
print(results_df)


Environment variable set!
Data downloaded successfully!
Data loaded! Shape: (28276227, 27)
Cleaning categorical variables...
Categorical variables cleaned.
Data split into train and test sets.
✅ feature_names.json saved!


NameError: name 'bucket' is not defined

In [ ]:
# Train, save, and upload models
for name, model in models.items():
    print(f"🚀 Training {name}...")

    # Save model locally based on model type
    if name == "CatBoost":
        local_model_path = f"models/{name.replace(' ', '_')}_model.cbm"
        model.save_model(local_model_path)
    elif name == "XGBoost":
        local_model_path = f"models/{name.replace(' ', '_')}_model.json"
        model.save_model(local_model_path)  # Native format for XGBoost
    else:
        local_model_path = f"models/{name.replace(' ', '_')}_model.pkl"
        joblib.dump(model, local_model_path)

    # Upload to GCS
    gcs_model_path = f"uk_property_data/models/{name.replace(' ', '_')}_model.{local_model_path.split('.')[-1]}"
    upload_to_gcs(local_model_path, gcs_model_path)
    print(f"✅ {name} model uploaded to GCS at: {gcs_model_path}")


🚀 Training Linear Regression...
✅ Model uploaded to GCS at: gs://boothill2001-dataset/uk_property_data/models/Linear_Regression_model.pkl
✅ Linear Regression model uploaded to GCS at: uk_property_data/models/Linear_Regression_model.pkl
🚀 Training XGBoost...
✅ Model uploaded to GCS at: gs://boothill2001-dataset/uk_property_data/models/XGBoost_model.json
✅ XGBoost model uploaded to GCS at: uk_property_data/models/XGBoost_model.json
🚀 Training Gradient Boosting...
✅ Model uploaded to GCS at: gs://boothill2001-dataset/uk_property_data/models/Gradient_Boosting_model.pkl
✅ Gradient Boosting model uploaded to GCS at: uk_property_data/models/Gradient_Boosting_model.pkl
🚀 Training LightGBM...
✅ Model uploaded to GCS at: gs://boothill2001-dataset/uk_property_data/models/LightGBM_model.pkl
✅ LightGBM model uploaded to GCS at: uk_property_data/models/LightGBM_model.pkl
🚀 Training CatBoost...
✅ Model uploaded to GCS at: gs://boothill2001-dataset/uk_property_data/models/CatBoost_model.cbm
✅ CatBoost

In [ ]:
# Function to predict using downloaded model
def predict_with_downloaded_model(model_name, local_model_path):
    try:
        if model_name == "CatBoost":
            model = CatBoostRegressor()
            model.load_model(local_model_path)
            sample_data = X_test.iloc[:5]
            predictions = model.predict(sample_data)
            print(f"\n🌟 Predictions with {model_name}: {predictions}")

        elif model_name == "XGBoost":
            model = Booster()
            model.load_model(local_model_path)
            sample_data = X_test.iloc[:5]
            dtest = DMatrix(sample_data)
            predictions = model.predict(dtest)
            print(f"\n🌟 Predictions with {model_name}: {predictions}")

        else:
            model = joblib.load(local_model_path)
            sample_data = X_test.iloc[:5]
            predictions = model.predict(sample_data)
            print(f"\n🌟 Predictions with {model_name}: {predictions}")
    except Exception as e:
        print(f"❌ Error predicting with {model_name}: {e}")

# Test prediction with uploaded models
model_names = ["Linear Regression", "XGBoost", "Gradient Boosting", "LightGBM", "CatBoost"]
for model_name in model_names:
    if model_name == "CatBoost":
        gcs_model_path = f"uk_property_data/models/{model_name.replace(' ', '_')}_model.cbm"
        local_model_path = f"models/{model_name.replace(' ', '_')}_model.cbm"
    elif model_name == "XGBoost":
        gcs_model_path = f"uk_property_data/models/{model_name.replace(' ', '_')}_model.json"
        local_model_path = f"models/{model_name.replace(' ', '_')}_model.json"
    else:
        gcs_model_path = f"uk_property_data/models/{model_name.replace(' ', '_')}_model.pkl"
        local_model_path = f"models/{model_name.replace(' ', '_')}_model.pkl"

    # Download the model from GCS
    download_from_gcs(gcs_model_path, local_model_path)

    # Predict using the downloaded model
    predict_with_downloaded_model(model_name, local_model_path)


Data already exists locally.

🌟 Predictions with Linear Regression: [11.6514126  11.98478613 12.21052512 11.20485409 11.26072069]
Data already exists locally.

🌟 Predictions with XGBoost: [12.271098 11.543489 12.907071 10.880221 11.117068]
Data already exists locally.

🌟 Predictions with Gradient Boosting: [12.29222846 11.53766225 12.90189687 10.89345442 11.15265201]
Data already exists locally.

🌟 Predictions with LightGBM: [12.26735371 11.54990588 12.90679752 10.88655759 11.1136388 ]
Data already exists locally.

🌟 Predictions with CatBoost: [12.26120159 11.54045624 12.90332883 10.88110816 11.12599917]


In [ ]:
upload_blob = bucket.blob("uk_property_data/models/feature_names.json")
upload_blob.upload_from_filename("models/feature_names.json")
print("✅ Đã upload feature_names.json lên GCS!")

✅ Đã upload feature_names.json lên GCS!


In [ ]:
# Upload feature_names.json to GCS
def upload_to_gcs(local_path, gcs_path):
    client = storage.Client()
    bucket = client.bucket(bucket_name)
    blob = bucket.blob(gcs_path)
    blob.upload_from_filename(local_path)
    print(f"✅ Uploaded {local_path} to gs://{bucket_name}/{gcs_path}")

upload_to_gcs("feature_names.json", "uk_property_data/models/feature_names.json")


✅ Uploaded feature_names.json to gs://boothill2001-dataset/uk_property_data/models/feature_names.json


In [ ]:
print("🧠 Final feature order for model input:")
print(list(X_train.columns))


🧠 Final feature order for model input:
['Year', 'Month', 'Quarter', 'Is_Weekend', 'High_Value_Property', 'Holiday_Season', 'Price_Change_Ratio', 'Anomaly', 'Property_Type_F', 'Property_Type_O', 'Property_Type_S', 'Property_Type_T', 'Season_Spring', 'Season_Summer', 'Season_Winter', 'Duration_L', 'Duration_U', 'PPDCategory_Type_B']


In [ ]:
# Tải lại để check
from google.cloud import storage
import json

client = storage.Client()
bucket = client.bucket("boothill2001-dataset")
blob = bucket.blob("uk_property_data/models/feature_names.json")
blob.download_to_filename("downloaded_feature_names.json")

with open("downloaded_feature_names.json") as f:
    features = json.load(f)

print("👀 Feature order in GCS file:", features)  # xem 10 cột đầu


👀 Feature order in GCS file: ['Year', 'Month', 'Quarter', 'Is_Weekend', 'High_Value_Property', 'Holiday_Season', 'Price_Change_Ratio', 'Anomaly', 'Property_Type_F', 'Property_Type_O', 'Property_Type_S', 'Property_Type_T', 'Season_Spring', 'Season_Summer', 'Season_Winter', 'Duration_L', 'Duration_U', 'PPDCategory_Type_B']


In [ ]:
upload_to_gcs("feature_names.json", "uk_property_data/models/feature_names.json")


✅ Uploaded feature_names.json to gs://boothill2001-dataset/uk_property_data/models/feature_names.json
